# 4. Interactive Dash Dashboard

Launch the Harmonia experiment dashboard for interactive exploration of results.

The dashboard has 5 tabs:
1. **Overview** — Runs table with summary stats
2. **Metrics** — Bar charts, heatmaps, scatter plots
3. **Trace Explorer** — Deep-dive into individual run traces
4. **Token & Cost** — Token usage and cost analysis
5. **Comparison** — Side-by-side run comparison

In [ ]:
import os, subprocess, socket, time, webbrowser
from pathlib import Path

os.chdir("/hpc/compgen/projects/llm_GEO_project/harmonia_metadata_agent/analysis/dstoker/harmonia")
PYTHON = ".venv/bin/python"
RESULTS_DIR = "results/"

## Check what data is available for the dashboard

In [ ]:
# List results directories with metrics.json
results_path = Path(RESULTS_DIR)
runs_with_metrics = []
runs_without_metrics = []
for d in sorted(results_path.iterdir()):
    if not d.is_dir() or d.name in ("old", "older"):
        continue
    if (d / "metrics.json").exists():
        runs_with_metrics.append(d.name)
    elif (d / "trace.json").exists():
        runs_without_metrics.append(d.name)

print(f"Runs with metrics.json (ready for dashboard): {len(runs_with_metrics)}")
for r in runs_with_metrics:
    print(f"  ✓ {r}")

if runs_without_metrics:
    print(f"\nRuns with trace but no metrics (need calculate_metrics.py): {len(runs_without_metrics)}")
    for r in runs_without_metrics:
        print(f"  ⚠ {r}")

## Find a free port and launch the dashboard

The dashboard binds to `0.0.0.0` so it's accessible from outside.
To access from your local machine, set up SSH port forwarding:

```bash
ssh -L 8050:<hostname>:8050 <hpc-login>
```

Then open http://localhost:8050 in your browser.

In [ ]:
def find_free_port(start=8050, end=8099):
    for port in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("", port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"No free port in {start}-{end}")

port = find_free_port()
hostname = socket.gethostname()
print(f"Will use port {port} on {hostname}")
print(f"\nTo access from your local machine, run:")
print(f"  ssh -L {port}:{hostname}:{port} <your-hpc-login>")
print(f"\nThen open: http://localhost:{port}")

In [ ]:
# Launch the dashboard as a background subprocess
# (It will keep running until you stop the kernel or kill the process)

proc = subprocess.Popen(
    [PYTHON, "src/dashboard/app.py",
     "--results-dir", RESULTS_DIR,
     "--port", str(port)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait a moment and show initial output
import time
time.sleep(3)

# Read any available output
import select
import io

print(f"Dashboard PID: {proc.pid}")
print(f"Dashboard should be starting on port {port}...")
print(f"\nAccess via: http://localhost:{port}")
print(f"SSH tunnel: ssh -L {port}:{hostname}:{port} <your-hpc-login>")
print(f"\nTo stop: proc.terminate() or restart this kernel")

## Check dashboard status

In [ ]:
# Check if the dashboard is still running
if proc.poll() is None:
    print(f"Dashboard is running (PID {proc.pid})")
    print(f"Access: http://localhost:{port}")
else:
    print(f"Dashboard exited with code {proc.returncode}")
    out = proc.stdout.read()
    print(out[-3000:] if len(out) > 3000 else out)

## Stop the dashboard

In [ ]:
# Uncomment to stop:
# proc.terminate()
# proc.wait()
# print("Dashboard stopped")